# Project 03 — Customer Churn / Sales Trend Analysis

**Objective:** Analyse an e-commerce sales dataset to identify revenue trends, best categories,
and valuable customers, then recommend actions.

**Data note:** This project uses a reproducible, simulated dataset generated for this exercise.
It contains no real customer information.

## 1. Business questions
1. Which months generate the most and least revenue?
2. Which product categories perform best?
3. What share of revenue comes from the top 10% of customers?
4. What actions should the business test next?

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_FILE = Path("ecommerce_orders.csv")
if not DATA_FILE.exists():
    rng = np.random.default_rng(42)
    n = 2000
    orders = pd.DataFrame({
        "order_date": pd.to_datetime("2025-01-01") + pd.to_timedelta(
            rng.integers(0, 365, n), unit="D"),
        "customer_id": rng.integers(1, 301, n),
        "category": rng.choice(
            ["Electronics", "Fashion", "Home", "Books"], n,
            p=[0.30, 0.35, 0.20, 0.15]),
        "amount": rng.gamma(2.0, 600, n).round(2)
    })
    orders.to_csv(DATA_FILE, index=False)

orders = pd.read_csv(DATA_FILE, parse_dates=["order_date"])
print("Dataset shape:", orders.shape)
display(orders.head())
print("\\nData types:")
display(orders.dtypes)
print("\\nMissing values:")
display(orders.isna().sum())
print("Duplicate rows:", orders.duplicated().sum())

## 2. Feature engineering and monthly trend
Extract month, weekday, and quarter. Calculate monthly revenue and month-over-month percentage change.

In [ ]:
orders["month"] = orders["order_date"].dt.to_period("M")
orders["month_start"] = orders["order_date"].dt.to_period("M").dt.to_timestamp()
orders["weekday"] = orders["order_date"].dt.day_name()
orders["quarter"] = orders["order_date"].dt.quarter

monthly = orders.groupby("month_start")["amount"].sum().sort_index()
monthly_growth = monthly.pct_change() * 100
monthly_results = pd.DataFrame({
    "revenue_usd": monthly,
    "month_over_month_growth_pct": monthly_growth
})
display(monthly_results.round(2))
print("Best month:", monthly.idxmax(), "| Lowest month:", monthly.idxmin())

## 3. Category analysis and pivot table

In [ ]:
by_category = (orders.groupby("category")["amount"]
               .agg(total_revenue="sum", average_order_value="mean", order_count="count")
               .sort_values("total_revenue", ascending=False))
display(by_category.round(2))

monthly_category = pd.pivot_table(
    orders, index="month_start", columns="category",
    values="amount", aggfunc="sum", fill_value=0
)
display(monthly_category.round(2))

## 4. Customer value — Pareto-style analysis
Calculate total spend by customer and measure the revenue share generated by the top 10% of customers.

In [ ]:
customer_spend = orders.groupby("customer_id")["amount"].sum().sort_values(ascending=False)
top_n = max(1, int(np.ceil(len(customer_spend) * 0.10)))
top_customers = customer_spend.head(top_n)
top_share = top_customers.sum() / customer_spend.sum()

print(f"Unique customers: {len(customer_spend)}")
print(f"Top 10% customer count: {top_n}")
print(f"Revenue from top 10%: ${top_customers.sum():,.2f}")
print(f"Top 10% revenue share: {top_share:.1%}")
display(top_customers.head(10))

## 5. Visual dashboard
The dashboard includes monthly revenue, category revenue, monthly category trends, and a customer concentration (Pareto-style) curve.

In [ ]:
sns.set_theme(style="whitegrid", palette="deep")
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
fig.suptitle("E-commerce Sales Analysis — 2025 Simulated Orders",
             fontsize=16, fontweight="bold")

axes[0, 0].plot(monthly.index, monthly.values, marker="o", linewidth=2)
axes[0, 0].scatter([monthly.idxmax()], [monthly.max()], s=90, label="Best month")
axes[0, 0].scatter([monthly.idxmin()], [monthly.min()], s=90, label="Lowest month")
axes[0, 0].set(title="Monthly revenue trend", xlabel="Month", ylabel="Revenue (USD)")
axes[0, 0].tick_params(axis="x", rotation=35)
axes[0, 0].legend()

cat = by_category["total_revenue"].sort_values()
axes[0, 1].barh(cat.index, cat.values)
axes[0, 1].set(title="Revenue by category", xlabel="Revenue (USD)", ylabel="Category")

for category in monthly_category.columns:
    axes[1, 0].plot(monthly_category.index, monthly_category[category],
                    marker=".", label=category)
axes[1, 0].set(title="Monthly revenue by category", xlabel="Month", ylabel="Revenue (USD)")
axes[1, 0].tick_params(axis="x", rotation=35)
axes[1, 0].legend(title="Category", fontsize=8)

cum_rev = customer_spend.cumsum() / customer_spend.sum() * 100
cum_customers = np.arange(1, len(customer_spend) + 1) / len(customer_spend) * 100
axes[1, 1].plot(cum_customers, cum_rev)
axes[1, 1].axvline(10, linestyle="--", label="Top 10% of customers")
axes[1, 1].axhline(top_share * 100, linestyle="--", label=f"Revenue share: {top_share:.1%}")
axes[1, 1].set(title="Customer revenue concentration",
               xlabel="Cumulative share of customers (%)",
               ylabel="Cumulative share of revenue (%)", xlim=(0, 100), ylim=(0, 100))
axes[1, 1].legend(fontsize=8)
plt.show()

## 6. Business recommendations

1. Pilot a retention offer for the top 10% of customers and compare results with a holdout group.
2. Prioritize review of stock availability and conversion in the highest-revenue category, while monitoring margins.
3. Investigate campaigns, inventory, and seasonality around the best and lowest revenue months before planning promotions.

These are hypotheses for testing, not claims about a real business. The dataset is simulated.

In [ ]:
# Export key tables for review and save the dashboard.
monthly_results.to_csv("monthly_revenue.csv")
by_category.to_csv("category_summary.csv")
customer_spend.rename("total_spend_usd").to_csv("customer_spending.csv")
monthly_category.to_csv("monthly_category_pivot.csv")
fig.savefig("sales_analysis_dashboard.png", dpi=150, bbox_inches="tight")
print("Saved dashboard and analysis CSV files.")